In [22]:
# DO NOT upgrade everything
# Only install what Colab doesn't already have

!pip install -q \
langchain-community \
langchain-text-splitters \
chromadb \
sentence-transformers \
langchain-groq

In [23]:
!pip install -q langchain

In [24]:
# ===================== TEXT SPLITTING =====================
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ===================== EMBEDDINGS =====================
from langchain_community.embeddings import HuggingFaceEmbeddings

# ===================== VECTOR STORE =====================
from langchain_community.vectorstores import Chroma

# ===================== LLM =====================
from langchain_community.llms import HuggingFacePipeline

# ===================== LCEL (NEW CHAIN SYSTEM) =====================
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ===================== PROMPT =====================
from langchain_core.prompts import PromptTemplate

# ===================== DOCUMENT =====================
from langchain_core.documents import Document

# ===================== TRANSFORMERS =====================
from transformers import AutoTokenizer, pipeline

# ===================== TORCH =====================
import torch

print("Imports successful ✅")

Imports successful ✅


In [25]:
# Step 3: Create sample documents (your knowledge base)
print("\nStep 1: Creating knowledge base...")
documents = [
    Document(page_content="LangChain is a framework for developing applications powered by language models. It provides tools for document loading, splitting, embeddings, and chains.",
             metadata={"source": "doc1"}),
    Document(page_content="RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retrieved based on similarity, then an LLM generates an answer using those documents as context.",
             metadata={"source": "doc2"}),
    Document(page_content="Vector databases store embeddings and allow for semantic similarity search. They convert text into numerical vectors that capture meaning.",
             metadata={"source": "doc3"}),
    Document(page_content="ChromaDB is an open-source embedding database that works well with LangChain. It's lightweight and perfect for development.",
             metadata={"source": "doc4"}),
    Document(page_content="HuggingFace provides thousands of pre-trained models that can run locally or in the cloud. Popular models include FLAN-T5, GPT-2, and Llama.",
             metadata={"source": "doc5"}),
    Document(page_content="Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings.",
             metadata={"source": "doc6"}),
]
print("\n all done")


Step 1: Creating knowledge base...

 all done


In [26]:
# Step 4: Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(documents)
print(f"Created {len(splits)} document chunks")

Created 6 document chunks


In [27]:
# Step 5: Create embeddings
print("\nStep 2: Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")


Step 2: Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


In [28]:
# Step 6: Create vector store
print("\n Step 3: Creating vector database...")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="rag_demo"
)
print(" Vector database created")

# Step 7: Set up retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}  # Retrieve top 2 most relevant chunks
)


 Step 3: Creating vector database...
 Vector database created


In [29]:
# ===================== LOAD LLM (GROQ - CLOUD) =====================

print("\nStep 4: Loading Language Model...")
print("... (Using Groq - llama-3.1-8b-instant)")

import os
from langchain_groq import ChatGroq

# Read your API key from the local environment
groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("Set GROQ_API_KEY in your environment before running this notebook")

# Create LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.7
)

print("Language model loaded (Groq) ✅")


Step 4: Loading Language Model...
... (Using Groq - llama-3.1-8b-instant)
Language model loaded (Groq) ✅


In [30]:
# ===================== STEP 9: CREATE PROMPT =====================

from langchain_core.prompts import PromptTemplate

template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Keep the answer concise and relevant.

Context:
{context}

Question:
{question}

Answer:"""

QA_PROMPT = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [31]:
# ===================== STEP 10: CREATE RAG CHAIN =====================

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("\nBuilding RAG chain...")

# Format retrieved documents into text
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create RAG pipeline
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | QA_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready! ✅")


Building RAG chain...
RAG chain ready! ✅


In [32]:
# ===================== STEP 11: TEST THE COMPLETE RAG SYSTEM =====================

print("\n" + "="*60)
print(" COMPLETE RAG DEMONSTRATION ")
print("="*60)


def ask_question(query):
    """
    Complete RAG pipeline: Retrieval + Augmented + Generation
    """

    print(f"\nQuestion: {query}")
    print("\nStep 1: Retrieving relevant documents...")

    # ✅ Manual retrieval (required in new LangChain)
    docs = retriever.invoke(query)
    print(f"Retrieved {len(docs)} documents:")

    for i, doc in enumerate(docs, start=1):
        snippet = doc.page_content[:150].replace("\n", " ")
        print(f"\n  {i}. {snippet}...")
        print(f"     Source: {doc.metadata.get('source', 'unknown')}")

    print("\nStep 2: Generating answer using LLM...")

    # ✅ Correct LCEL execution
    answer = qa_chain.invoke(query)

    print(f"\nAnswer: {answer}")
    print("\n" + "-" * 60)


# ===================== RUN DEMO QUERIES =====================

print("\nRunning Demo Queries...\n")

ask_question("What is RAG?")
ask_question("What are embeddings and why are they useful?")
ask_question("Which database should I use for vector storage?")


 COMPLETE RAG DEMONSTRATION 

Running Demo Queries...


Question: What is RAG?

Step 1: Retrieving relevant documents...
Retrieved 2 documents:

  1. RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retriev...
     Source: doc2

  2. RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation. First, relevant documents are retriev...
     Source: doc2

Step 2: Generating answer using LLM...

Answer: RAG stands for Retrieval-Augmented Generation. It combines retrieval of relevant documents with text generation.

------------------------------------------------------------

Question: What are embeddings and why are they useful?

Step 1: Retrieving relevant documents...
Retrieved 2 documents:

  1. Embeddings are numerical representations of text that capture semantic meaning. Similar texts have similar embeddings....
     Source: doc6

  2. Em